# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Ramam Agarwal
**`Roll Number`:** U20230069
**`GitHub Branch`:** Ramam_U20230069  

# Imports and Setup

In [2]:
pip install rlcmab_sampler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [32]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())

                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [33]:
# Inspecting the provided dataset
print(train_users.isnull().sum())
print ('----------------------------------------------------------')
print(test_users.isnull().sum())
print ('----------------------------------------------------------')
print(news_df.isnull().sum())
print ('----------------------------------------------------------')
print(train_users.dtypes)
print ('----------------------------------------------------------')
print(test_users.dtypes)
print ('----------------------------------------------------------')
print(news_df.dtypes)

user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count             0
session_inactivity_d

In [34]:
# Handling missing values
train_age_mean = train_users['age'].mean()
test_age_mean = test_users['age'].mean()

train_users['age'] = train_users['age'].fillna(train_age_mean)
test_users['age'] = test_users['age'].fillna(test_age_mean)

print("Missing 'age' values in train_users after imputation:", train_users['age'].isnull().sum())
print("Missing 'age' values in test_users after imputation:", test_users['age'].isnull().sum())

Missing 'age' values in train_users after imputation: 0
Missing 'age' values in test_users after imputation: 0


In [35]:
# Handling missing values
news_df['headline'] = news_df['headline'].fillna('Unknown Headline')
news_df['short_description'] = news_df['short_description'].fillna('No Description')
news_df['authors'] = news_df['authors'].fillna('Unknown Author')

print("Missing values after handling:")
print(news_df.isnull().sum())

Missing values after handling:
link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
dtype: int64


In [36]:
target_col = 'label'

# Separate Features (X) and Target (y)
X = train_users.drop(['user_id', target_col], axis=1)
y = train_users[target_col]

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Encoding categorical columns: {list(categorical_cols)}")

# Encode Categorical Features
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Encode Target Labels (User1 -> 0, User2 -> 1, User3 -> 2)
target_le = LabelEncoder()
y_encoded = target_le.fit_transform(y)

Encoding categorical columns: ['browser_version', 'region_code']


In [37]:
# Encoding categorical column
from sklearn.preprocessing import LabelEncoder

le_category = LabelEncoder()
news_df['category_encoded'] = le_category.fit_transform(news_df['category'])

print("News_df after category encoding:")
print(news_df.head())

News_df after category encoding:
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenf

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [40]:
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_val:", X_val.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_val:", y_val.shape)

Shape of X_train: (1600, 31)
Shape of X_val: (400, 31)
Shape of y_train: (1600,)
Shape of y_val: (400,)


In [46]:
# Training XGBoost Classifier (Gradient Boosting)
import xgboost as xgb
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1200,
    learning_rate=0.01,
    max_depth=5,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0.2,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1200, n_jobs=None, num_class=3, ...)

In [47]:
from sklearn.metrics import accuracy_score, classification_report
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("-" * 30)
print(f"Validation Accuracy: {accuracy:.4f}")
print("-" * 30)
print("\nClassification Report:\n")
print(classification_report(y_val, y_pred, target_names=target_le.classes_))

------------------------------
Validation Accuracy: 0.8750
------------------------------

Classification Report:

              precision    recall  f1-score   support

      user_1       0.89      0.81      0.85       147
      user_2       0.95      0.88      0.92       141
      user_3       0.78      0.96      0.86       112

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.87       400
weighted avg       0.88      0.88      0.88       400



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
